In [1]:
import requests
from datetime import datetime
import os
from dotenv import load_dotenv
import pandas as pd
import json
import re
import html
import numpy as np
load_dotenv()
api_key = os.getenv("API_KEY")

In [6]:
class ExtractChampionData:

    def __init__(self, version : int = 0):
        self.last_version = self.get_latest_version(version=version)
        self.list_champ = self.get_all_champ_general_data()
        self.list_champ = list(self.list_champ.keys())

    def get_latest_version(self, version : int = 0):
        """Get the last version of the game"""
        url = "https://ddragon.leagueoflegends.com/api/versions.json"
        rep = requests.get(url).json()
        latest = rep[version]
        return latest

    def get_all_champ_general_data(self):
        """Get the champion data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion.json"
        resp = requests.get(url).json()
        return resp['data']
    
    def get_details_champ_data(self):
        """Get detail champion data"""
        data = []
        for champ in self.list_champ:
            url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/champion/{champ}.json"
            resp = requests.get(url).json()['data']
            data.append(resp[champ])
        return data
    
    def get_runes(self):
        """Get the runes data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/runesReforged.json"
        resp = requests.get(url).json()
        return resp
    
    def get_items(self):
        """Get the items data"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/item.json"
        resp = requests.get(url).json()
        return resp
    
    def get_summoner_seplls(self):
        """Get the summoner spells"""
        url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/data/fr_FR/summoner.json"
        resp = requests.get(url).json()
        return resp
    
    # def download_all_png_champion(self):
    #     """Download all champions pictures"""
    #     for champ in self.list_champ:
    #         url = f"https://ddragon.leagueoflegends.com/cdn/{self.last_version}/img/champion/{champ}.png"
    #         path_save = fr"C:\Users\najim\Documents\Projets\LeagueOfLegends\images\{champ}.png"

    #         resp = requests.get(url)
    #         with open(path_save, "wb") as file:
    #             file.write(resp.content)

In [12]:
class TransformChampionData:

    def __init__(self):
        self.keys_to_drop = ['image', 'skins', 'blurb']
        self.table_champion = ['key', 'name', 'title', 'lore', 'tags', 'partype']
        self.table_champ_passive = ['key', 'passive']
        self.table_champ_info = ['key', 'info', 'allytips', 'enemytips']
        self.table_champ_spells = ['key', 'spells']
        self.table_champ_stats = ['key', 'stats']
    
    def drop_keys(self, data : list) -> list:
        new_data = []
        for dict_champ in data:
            dict_champ = {k: v for k,v in dict_champ.items() if k not in self.keys_to_drop}
            new_data.append(dict_champ)
        return new_data
    
    def dispatch_data(self, data : list):
        table_champion = []
        table_champ_passive = []
        table_champ_info = []
        table_champ_spells = []
        table_champ_stats = []
        
        for dict_champ in data:
            champion = {k: v for k,v in dict_champ.items() if k in self.table_champion}
            champ_passive = {k: v for k,v in dict_champ.items() if k in self.table_champ_passive}
            champ_info = {k: v for k,v in dict_champ.items() if k in self.table_champ_info}
            champ_spells = {k: v for k,v in dict_champ.items() if k in self.table_champ_spells}
            champ_stats = {k: v for k,v in dict_champ.items() if k in self.table_champ_stats}

            table_champion.append(champion)
            table_champ_passive.append(champ_passive)
            table_champ_info.append(champ_info)
            table_champ_spells.append(champ_spells)
            table_champ_stats.append(champ_stats)
            
        return table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats
    
    def split_stats(self, data : list):
        stats_cst, stats_up = [], []
        for stat in data:
            cst = {k:v for k,v in stat.items() if "perlevel" not in k}
            up = {k: v for k, v in stat.items() if "perlevel" in k or "key" in k}
            stats_cst.append(cst)
            stats_up.append(up)
        return stats_cst, stats_up

    def listing_into_text(self, data : list, key : str, sep : str = " "):
        new_data = []
        for champ in data:
            new_value = f"{sep}".join(champ[key])
            champ[key] = self._clean_text(new_value)
            new_data.append(champ)
        return new_data

    def dict_into_first_level(self, data : list, key_to_flat : str) -> list:
        new_data = []
        for champ in data:
            base = {k: v for k, v in champ.items() if k != key_to_flat}
            base.update(champ.get(key_to_flat, {}))
            new_data.append(base)
        return new_data
    
    def pop_key(self, data : list, key_to_remove : list[str]) -> list:
        new_data = []
        for i in data:
            for key in key_to_remove:
                d = i.pop(key, None)
            new_data.append(i)
        return new_data
    
    def _clean_text(self, desc : str) -> str:
        desc = re.sub(r"<[^>]+>|<[a-z]>", " ", desc)
        desc = re.sub(r"\xa0", " ", desc)
        desc = re.sub(r"_ClientTooltipWrapper|Wrapper", "", desc)
        desc = re.sub(r"\{\{.*?\}\}", "XX", desc)
        desc = re.sub(r"XX$", "", desc)
        desc = re.sub(r"[ ]+", " ", desc)
        return desc

    def text_cleaning(self, data : list, key : str) -> list:
        clean = []
        for champ in data:
            desc = self._clean_text(champ[key])
            champ[key] = desc
            clean.append(champ)
        return clean
    
    def rename_cols(self, df : pd.DataFrame, rename : dict) -> pd.DataFrame:
        return df.rename(columns=rename)

    def clean_spells_data(self, data: list):
        new_data = []
        for champ in data:
            idchamp = champ['key']
            spells = champ['spells']

            spells = self.pop_key(
                spells,
                key_to_remove=['leveltip', 'description', 'cooldown', 'cost', 'datavalues',
                    'effect', 'vars', 'image', 'effectBurn', 'range', 'resource'])

            spells_clean = self.text_cleaning(spells, 'tooltip')
            spells_clean = self.text_cleaning(spells, 'name')
            spells_clean = self.text_cleaning(spells, 'id')

            for spell in spells_clean:
                spell['key'] = idchamp
                new_data.append(spell)

        return new_data
    
    def correct_spell_name(self, df_spells : pd.DataFrame, df_champ : pd.DataFrame) -> pd.DataFrame:
        """Rectifie les noms des spells de chaque champion"""

        df_spells = df_spells.rename(columns={'name':'spell_name'})
        df_merge = pd.merge(df_spells, df_champ[['key', 'name']], on='key')
        letters = ['Q', 'W', 'E', 'R']

        # numérotation 0,1,2,3 par groupe de 'name'
        df_merge['spell_rank'] = df_merge.groupby('name').cumcount()
        # on mappe 0→Q, 1→W, 2→E, 3→R
        df_merge['spell_id'] = df_merge['name'] + df_merge['spell_rank'].map(lambda x: letters[x])
        df_merge['spell_rank'] = df_merge['spell_rank'] + 1
        df_merge = df_merge.drop(columns=['name', 'id'])
        df_merge = df_merge.rename(columns={'key':'champ_id', 'cooldownBurn':'cooldown_burn', 
                                            'costBurn':'cost_burn', 'costType':'cost_type', 'rangeBurn':'range_burn'})

        df_merge = df_merge[['champ_id', 'spell_id', 'spell_name', 'tooltip', 'maxrank', 'cooldown_burn', 'cost_burn', 
                             'cost_type', 'maxammo', 'range_burn', 'spell_rank', 'patch_id']]
        return df_merge
        
    def correct_spell_costype(self, df_spells : pd.DataFrame, df_champ : pd.DataFrame) -> pd.DataFrame:
        """Rectifie les resources consommées de chaque spells"""

        df_merge = pd.merge(df_spells, df_champ[['key', 'partype']], left_on='champ_id', right_on='key')

        df_merge['cost_type'] = df_merge['cost_type'].str.strip()

        df_merge['cost_type'] = np.where(
            df_merge['cost_type'] == r"{{ abilityresourcename }}", 
            df_merge['partype'], 
            df_merge['cost_type'])
        
        expression = r"\(?\{\{.*?\}\}\)?|\+|\."
        df_merge["cost_type"] = df_merge["cost_type"]\
            .str.replace(expression, "", regex=True)\
            .str.replace(r"\s+", ' ', regex=True)\
            .str.lower()

        return df_merge.drop(columns=['key', 'partype'])

    def transform_to_df(self, data : list, version : str) -> pd.DataFrame:
        """Transforme une liste de dictionnaire en dataframe pandas"""
        data = pd.DataFrame(data)
        data['patch_id'] = int(re.sub(r'[^0-9]', "", version))
        return data
    
    def patch_table(self, version : str) -> pd.DataFrame:
        df = [{'id' : int(re.sub(r'[^0-9]', "", version)),
              'version' : version,
              'is_latest' : True}]
        return pd.DataFrame(df)
    
    def transform_runes(self, data : list, version : str) -> pd.DataFrame:
        """Récupérer les runes et transforme en dataframe pandas"""
        df_runes = pd.DataFrame(data)\
            .drop(columns=['icon', 'key'])\
            .rename(columns={'id' : 'type_rune_id', 'name':'type_name'})
        
        df_slots = df_runes.explode('slots').reset_index(drop=True)

        df_slots = pd.concat([
            df_slots.drop(columns=['slots']), 
            pd.json_normalize(df_slots['slots'])], 
            axis=1)

        df_runes = df_slots.explode('runes').reset_index(drop=True)
        
        df_runes = pd.concat([
            df_runes.drop(columns=['runes']), 
            pd.json_normalize(df_runes['runes'])], 
            axis=1)
        
        df_runes = df_runes\
            .rename(columns={'id':'child_rune_id', 'longDesc' : 'description'})\
            .drop(columns=['icon', 'key', 'shortDesc'])
        
        df_runes['description'] = [self._clean_text(x).strip() for x in df_runes['description']]
        df_runes['patch_id'] = re.sub(r"[^0-9]", "", version)
        
        return df_runes
    
    def transform_items(self, version : str):
        keys_to_drop = ['colloq','into','image','maps','stats', 'from', 'depth', 'inStore',	'effect',	
                        'consumed',	'stacks', 'hideFromAll', 'consumeOnFull', 'specialRecipe', 'requiredChampion']
        all_items = []
        items = ExtractChampionData().get_items()
        items = items['data']
        for ids in items:
            dico = items[ids]
            dico = {k:v for k,v in dico.items() if k not in keys_to_drop}
            cost = dico['gold']['total']
            sell = dico['gold']['sell']
            dico.update({'cost':cost, 'sell':sell, 'item_id':ids})
            all_items.append(dico)
        
        all_items = self.listing_into_text(all_items, key='tags', sep=' / ')
        all_items = self.text_cleaning(all_items, key='description')
        all_items = self.transform_to_df(all_items, version=version).drop(columns=['gold'])
        all_items = self.rename_cols(all_items, rename={'description':'stats', 'plaintext':'description'})
        return all_items
    
    def transform_summoner(self, version : str):
        keys_to_keep = ['name', 'description', 'cooldownBurn', 'key']
        all_summoners = []
        summoners = ExtractChampionData().get_summoner_seplls()
        data = summoners['data']
        for summon in data:
            dico = data[summon]
            dico = {k:v for k,v in dico.items() if k in keys_to_keep}
            all_summoners.append(dico)
        
        all_summoners = self.transform_to_df(all_summoners, version=version)
        all_summoners = self.rename_cols(all_summoners, rename={'cooldownBurn':'cooldown_burn', 'key':'summoner_spell_id'})
        return all_summoners

In [ ]:
def pipeline_champion(version : int = 0):
    """Pipeline d'application des méthodes d'extraction, de transformation et de chargement"""
    
    extract = ExtractChampionData(version=version)
    lastest_version = extract.last_version
    print("Lastest version :", lastest_version)
    transform = TransformChampionData()

    details = extract.get_details_champ_data()
    table_runes = extract.get_runes()
    details = transform.drop_keys(details)
    
    table_champion, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats = transform.dispatch_data(details)

    
    table_champion = transform.listing_into_text(table_champion, "tags", sep=' / ')
    table_champion = transform.text_cleaning(table_champion, key='lore')
    table_champion = transform.text_cleaning(table_champion, key='title')
    
    table_champ_info = transform.listing_into_text(table_champ_info, key="enemytips", sep=' ')
    table_champ_info = transform.listing_into_text(table_champ_info, key="allytips", sep=' ')
    table_champ_info = transform.dict_into_first_level(table_champ_info, key_to_flat='info')

    table_champ_passive = transform.dict_into_first_level(table_champ_passive, key_to_flat='passive')
    table_champ_passive = transform.pop_key(table_champ_passive, key_to_remove=['image'])
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='name')
    table_champ_passive = transform.text_cleaning(table_champ_passive, key='description')

    table_champ_stats = transform.dict_into_first_level(table_champ_stats, key_to_flat='stats')
    table_champ_stats, table_champ_stats_up = transform.split_stats(table_champ_stats)

    table_champ_spells = transform.clean_spells_data(table_champ_spells)

    table_champion = transform.transform_to_df(table_champion, version=lastest_version)
    table_champ_info = transform.transform_to_df(table_champ_info, version=lastest_version)
    table_champ_passive = transform.transform_to_df(table_champ_passive, version=lastest_version)
    table_champ_stats = transform.transform_to_df(table_champ_stats, version=lastest_version)
    table_champ_stats_up = transform.transform_to_df(table_champ_stats_up, version=lastest_version)
    table_champ_spells = transform.transform_to_df(table_champ_spells, version=lastest_version)

    table_champ_spells = transform.correct_spell_name(table_champ_spells, table_champion)
    table_champ_spells = transform.correct_spell_costype(table_champ_spells, table_champion)

    table_champion_version = table_champion[['key', 'title', 'lore', 'tags', 'partype', 'patch_id']]
    table_champion = table_champion[['key', 'name']]

    table_runes = transform.transform_runes(table_runes, version=lastest_version)

    table_patch = transform.patch_table(version = lastest_version)

    table_item = transform.transform_items(version=lastest_version)

    table_champion = transform.rename_cols(table_champion, rename={'key':'id'})
    table_champion_version = transform.rename_cols(table_champion_version, rename={'key':'champ_id'})
    table_champ_info = transform.rename_cols(table_champ_info, rename={'key':'champ_id'})
    table_champ_passive = transform.rename_cols(table_champ_passive, rename={'key':'champ_id'})
    table_champ_stats = transform.rename_cols(table_champ_stats, rename={'key':'champ_id'})
    table_champ_stats_up = transform.rename_cols(table_champ_stats_up, rename={'key':'champ_id', 'hpperlevel':'hp_up',	'mpperlevel':'mp_up', 'armorperlevel':'armor_up',
                                                                            'spellblockperlevel':'spellblock_up', 'hpregenperlevel':'hpregen_up', 'mpregenperlevel':'mpregen_up',
                                                                            'critperlevel':'crit_up', 'attackdamageperlevel':'attackdamage_up',	'attackspeedperlevel':'attackspeed_up'})
    
    return table_champion, table_champion_version, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up, table_runes, table_patch, table_item

table_champion, table_champion_version, table_champ_passive, table_champ_info, table_champ_spells, table_champ_stats, table_champ_stats_up, table_runes, table_patch, table_item = pipeline_champion()

Lastest version : 16.4.1


In [5]:
def get_max_length(col : pd.Series):
    x = 0
    for i in col.astype(str):
        if len(i) > x:
            x = len(i)
    return x

def create_schema(table : pd.DataFrame):
    schema = []
    for i in table.columns:
        name = i
        col_type = table[i].dtype.name
        max_length = get_max_length(table[i])
        schema.append({
            'colonne' : name,
            'type' : col_type,
            'longueur' : max_length
        })
    return pd.DataFrame(schema)

In [13]:
r = TransformChampionData().transform_summoner('16.4.1')
r

,name,description,cooldown_burn,summoner_spell_id,patch_id
0,Barrière,Vous gagnez brièvement un bouclier.,180,21,1641
1,Purge,Dissipe toutes les entraves (sauf les neutrali...,240,1,1641
2,Saut éclair,Téléporte votre champion sur une courte distan...,0.25,2202,1641
3,Fuite,Vous gagnez un bref bonus en vitesse de déplac...,45,2201,1641
4,Embrasement,Inflige des dégâts bruts sur la durée au champ...,180,14,1641
5,Fatigue,Ralentit le champion ennemi ciblé et réduit le...,240,3,1641
6,Saut éclair,Vous téléporte sur une courte distance vers l'...,300,4,1641
7,Fantôme,Vous gagnez de la vitesse de déplacement et ig...,240,6,1641
8,Soins,Rend des PV à votre champion et au champion al...,240,7,1641
9,Clarté,Rend du mana à votre champion et aux champions...,240,13,1641


In [52]:
df_items = TransformChampionData().transform_items(version='16.4.1')
df_items

,name,stats,description,tags,cost,sell,item_id,patch_id
0,Bottes,+25 vitesse de déplacement,Augmente légèrement la vitesse de déplacement.,Boots,300,210,1001,1641
1,Charme féérique,+50% de régénération de base du mana,Augmente légèrement la régénération du mana.,ManaRegen,200,140,1004,1641
2,Collier rafraîchissant,+100% de régénération de base des PV,Augmente légèrement la régénération des PV.,HealthRegen,300,120,1006,1641
3,Ceinture du géant,+350 PV,Augmente grandement les PV.,Health,900,630,1011,1641
4,Cape d'agilité,+15% de chances de coup critique,Augmente les chances de coup critique.,CriticalStrike,600,420,1018,1641
...,...,...,...,...,...,...,...,...
683,Frappe du grizzly,"Illaoi frappe le sol avec son idole, infligea...",,,0,0,9405,1641
684,Ricochet des amants,"Désormais, Xayah lance aussi une dague spécia...",,,0,0,9406,1641
685,Maléfice renforcé,Aurora lance désormais plus de maléfices et l...,,,0,0,9407,1641
686,Fracasse-carotte,Riven gagne passivement de la vitesse de dépl...,,,0,0,9408,1641


In [6]:
display(table_champion.head(5))
display(schema_info := create_schema(table_champion))

,id,name
0,266,Aatrox
1,103,Ahri
2,84,Akali
3,166,Akshan
4,12,Alistar


,colonne,type,longueur
0,id,str,3
1,name,str,15


In [7]:
display(table_champion_version.head(5))
display(schema_info := create_schema(table_champion_version))

,champ_id,title,lore,tags,partype,patch_id
0,266,Épée des Darkin,"Autrefois, Aatrox et ses frères étaient honoré...",Fighter,Puits de sang,1641
1,103,Renarde à neuf queues,"Connectée à la magie du royaume spirituel, Ahr...",Mage / Assassin,Mana,1641
2,84,Assassin rebelle,Ayant abandonné l'Ordre Kinkou et le titre de ...,Assassin,Énergie,1641
3,166,Sentinelle rebelle,"Se jouant du danger, Akshan combat le mal sans...",Marksman / Assassin,Mana,1641
4,12,Minotaure,Alistar est un guerrier redoutable cherchant à...,Tank / Support,Mana,1641


,colonne,type,longueur
0,champ_id,str,3
1,title,str,27
2,lore,str,705
3,tags,str,19
4,partype,str,14
5,patch_id,int64,4


In [8]:
display(table_champ_info.head(5))
display(schema_info := create_schema(table_champ_info))

,champ_id,allytips,enemytips,attack,defense,magic,difficulty,patch_id
0,266,Utilisez Ruée obscure tout en lançant Épée des...,Les attaques d'Aatrox sont prévisibles. Profit...,8,4,3,4,1641
1,103,"Utilisez Charme pour préparer vos combos, cela...",Les capacités de survie d'Ahri sont considérab...,3,4,8,5,1641
2,84,Akali peut facilement tuer les champions fragi...,"Quand Akali est occultée par Linceul nébuleux,...",5,3,8,7,1641
3,166,"Se jouant du danger, Akshan combat le mal sans...",,0,0,0,0,1641
4,12,Atomisation peut vous aider à mieux vous place...,"Alistar peut être dangereux, mais il est solid...",6,9,5,7,1641


,colonne,type,longueur
0,champ_id,str,3
1,allytips,str,800
2,enemytips,str,584
3,attack,int64,2
4,defense,int64,2
5,magic,int64,2
6,difficulty,int64,2
7,patch_id,int64,4


In [9]:
display(table_champ_passive.head(5))
display(schema_info := create_schema(table_champ_passive))

,champ_id,name,description,patch_id
0,266,Posture du massacreur,"Régulièrement, la prochaine attaque de base d'...",1641
1,103,Vol essentiel,"Après avoir tué 9 sbires ou monstres, Ahri réc...",1641
2,84,Marque d'assassin,Blesser un champion avec une compétence crée u...,1641
3,166,Fourberie,Tous les trois coups venant de ses attaques ou...,1641
4,12,Cri triomphant,Alistar charge son cri en étourdissant ou en d...,1641


,colonne,type,longueur
0,champ_id,str,3
1,name,str,31
2,description,str,520
3,patch_id,int64,4


In [10]:
display(table_champ_spells.head(5))
display(schema_info := create_schema(table_champ_spells))

,champ_id,spell_id,spell_name,tooltip,maxrank,cooldown_burn,cost_burn,cost_type,maxammo,range_burn,spell_rank,patch_id
0,266,AatroxQ,Épée des Darkin,"Aatrox abat son épée, infligeant XX pts de dég...",5,14/12/10/8/6,0,pas de coût,-1,25000,1,1641
1,266,AatroxW,Chaînes infernales,"Aatrox lance une chaîne, ralentissant le premi...",5,20/18/16/14/12,0,pas de coût,-1,825,2,1641
2,266,AatroxE,Ruée obscure,Passive : Aatrox récupère des PV équivalents ...,5,9/8/7/6/5,0,pas de coût,-1,25000,3,1641
3,266,AatroxR,Fossoyeur des mondes,"Aatrox révèle sa vraie forme démoniaque, effra...",3,120/100/80,0,pas de coût,-1,25000,4,1641
4,103,AhriQ,Orbe d'illusion,"Ahri lance son orbe et le ramène vers elle, in...",5,7,55/65/75/85/95,mana,-1,970,1,1641


,colonne,type,longueur
0,champ_id,str,3
1,spell_id,str,16
2,spell_name,str,40
3,tooltip,str,982
4,maxrank,int64,1
5,cooldown_burn,str,22
6,cost_burn,str,19
7,cost_type,str,39
8,maxammo,str,2
9,range_burn,str,24


In [11]:
display(table_champ_stats.head(5))
display(schema_info := create_schema(table_champ_stats))

,champ_id,hp,mp,movespeed,armor,spellblock,attackrange,hpregen,mpregen,crit,attackdamage,attackspeed,patch_id
0,266,650,0,345,38,32,175,3.00,0.0,0,60,0.651,1641
1,103,590,418,330,21,30,550,2.50,8.0,0,53,0.668,1641
2,84,600,200,345,23,37,125,9.00,50.0,0,62,0.625,1641
3,166,610,350,330,26,30,500,3.75,8.2,0,52,0.638,1641
4,12,685,350,330,40,32,125,8.50,8.5,0,62,0.625,1641


,colonne,type,longueur
0,champ_id,str,3
1,hp,int64,3
2,mp,int64,5
3,movespeed,int64,3
4,armor,int64,2
5,spellblock,int64,2
6,attackrange,int64,3
7,hpregen,float64,4
8,mpregen,float64,5
9,crit,int64,1


In [12]:
display(table_champ_stats_up.head(5))
display(schema_info := create_schema(table_champ_stats_up))

,champ_id,hp_up,mp_up,armor_up,spellblock_up,hpregen_up,mpregen_up,crit_up,attackdamage_up,attackspeed_up,patch_id
0,266,114,0.0,4.8,2.05,0.50,0.0,0,5.00,2.500,1641
1,103,104,25.0,4.2,1.30,0.60,0.8,0,3.00,2.200,1641
2,84,119,0.0,4.7,2.05,0.90,0.0,0,3.30,3.200,1641
3,166,107,40.0,4.7,1.30,0.65,0.7,0,3.00,4.000,1641
4,12,120,40.0,4.7,2.05,0.85,0.8,0,3.75,2.125,1641


,colonne,type,longueur
0,champ_id,str,3
1,hp_up,int64,3
2,mp_up,float64,4
3,armor_up,float64,4
4,spellblock_up,float64,4
5,hpregen_up,float64,4
6,mpregen_up,float64,4
7,crit_up,int64,1
8,attackdamage_up,float64,6
9,attackspeed_up,float64,5


In [13]:
display(table_runes.head(5))
display(schema_info := create_schema(table_runes))

,type_rune_id,type_name,child_rune_id,name,description,patch_id
0,8100,Domination,8112,Électrocution,Toucher un champion ennemi avec 3 attaques ou ...,1641
1,8100,Domination,8128,Moisson noire,Blesser un champion qui a moins de 50% de ses ...,1641
2,8100,Domination,9923,Déluge de lames,"Quand vous attaquez un champion ennemi, votre ...",1641
3,8100,Domination,8126,Coup bas,Blesser des champions dont les actions ou les ...,1641
4,8100,Domination,8139,Goût du sang,Vous récupérez des PV quand vous blessez un ch...,1641


,colonne,type,longueur
0,type_rune_id,int64,4
1,type_name,str,11
2,child_rune_id,int64,4
3,name,str,27
4,description,str,575
5,patch_id,str,4


In [14]:
table_patch

,id,version,is_latest
0,1641,16.4.1,True
